# RNN Spoken Digit Classifier — Complete Training Notebook

This notebook runs all four tasks in order:

| Task | Model | Constraint | Purpose |
|------|-------|------------|---------|
| **A** | GRU (hidden=128) | None | Baseline teacher |
| **B1** | MGU (hidden=50, float32) | 36 kB / layer (float32) | Memory-constrained student |
| **B2** | MGU + INT8 QAT | 36 kB / layer (int8) | INT8 quantization-aware training |
| **C** | MGU + PoT QAT | 36 kB / layer (int8) | Power-of-Two weights |

**Requirements before running:**
- Upload your project `.zip` in Section 0
- Ensure a GPU runtime (`Runtime → Change runtime type → T4 GPU`)

## Section 0 — Setup

Clone the Free Spoken Digit Dataset, install dependencies, upload the project, and verify the environment.

In [ ]:
# 0.1 Clone FSDD and install dependencies
!git clone https://github.com/Jakobovski/free-spoken-digit-dataset.git
!pip install -q torch torchaudio librosa scikit-learn numpy pandas matplotlib

In [ ]:
# 0.2 Upload project zip from your local machine
from google.colab import files
uploaded = files.upload()   # select your project .zip

In [ ]:
# 0.3 Extract the zip (handles Windows backslash paths inside the archive)
import zipfile, os, pathlib

zip_name = list(uploaded.keys())[0]
extract_root = "/content/project"

with zipfile.ZipFile(zip_name, 'r') as zf:
    for member in zf.infolist():
        # Fix Windows paths: replace \ with /
        safe_path = member.filename.replace('\\', '/')
        target = os.path.join(extract_root, safe_path)
        if member.is_dir():
            os.makedirs(target, exist_ok=True)
        else:
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with zf.open(member) as src, open(target, 'wb') as dst:
                dst.write(src.read())

print(f"Extracted to {extract_root}")

In [ ]:
# 0.4 GPU check — must be a GPU runtime
import torch
assert torch.cuda.is_available(), "No GPU found! Change runtime type to T4 GPU."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
# 0.5 Verify project structure
import os
project_dir = "/content/project"
for root, dirs, files_list in os.walk(project_dir):
    dirs[:] = [d for d in sorted(dirs) if not d.startswith('.')]
    level = root.replace(project_dir, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    sub = '  ' * (level + 1)
    for f in sorted(files_list):
        if f.endswith(('.py', '.ipynb', '.json', '.pt')):
            print(f"{sub}{f}")

In [ ]:
# 0.6 Move into project directory and run diagnostic
# x_std (non-padded, all) must be in [0.9, 1.1] — diagnostic prints PASS or FAIL
%cd /content/project
!python diagnose_b1.py

**Expected diagnostic output:**
- `x std (non-padded, all)` in range `[0.9, 1.1]`
- `NORMALISATION CHECK: PASS`
- Loss trajectory should decrease from ~2.3 toward ~2.0 within 20 steps

If you see `FAIL`, check that `utils/data_preprocessing.py` uses **pre-padding** (zeros at the start, real audio at the end).

## Section 1 — Task A: Baseline GRU Teacher

Train the unconstrained GRU (hidden=128, N_MFCC=40).  
This produces `best_model.pt` which is used as the teacher for Tasks B1, B2, and C.

In [ ]:
# Train Task A (50 epochs, ~5-10 min on GPU)
!python main.py

In [ ]:
# Verify checkpoint exists
import os
assert os.path.exists('best_model.pt'), "best_model.pt not found — Task A must complete first"
print(f"best_model.pt  {os.path.getsize('best_model.pt'):,} bytes")

## Section 2 — Task B1: Memory-Constrained MGU (float32)

Train the Minimal Gated Unit student with knowledge distillation from Task A.  
Every layer must fit in 36 kB at float32 (9,000 params max per layer).  
The memory audit runs automatically before training and aborts on violation.

In [ ]:
# Train Task B1 (60 epochs, ~10-15 min on GPU)
!python task_b1_constrained.py

In [ ]:
# Show per-layer memory breakdown for B1
import sys; sys.path.insert(0, '.')
from task_b1_constrained import ConstrainedMGU
from utils.memory_utils import audit_model_memory
m = ConstrainedMGU()
audit_model_memory(m, bytes_per_param=4)   # float32

## Section 3 — Task B2: INT8 Quantization-Aware Training

Fake-quantize every `nn.Linear` weight to INT8 (symmetric, per-tensor) using a
Straight-Through Estimator.  Warm-starts from the B1 checkpoint.  
At INT8 the per-layer limit rises to 36,000 params (from 9,000 at float32).

In [ ]:
# Train Task B2 (60 epochs, ~15-20 min on GPU)
!python task_b2_int8.py

In [ ]:
# Plot B2 training curves from history_b2.json
import json, matplotlib.pyplot as plt

with open('history_b2.json') as f:
    h = json.load(f)

epochs = list(range(1, len(h['train_loss']) + 1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, h['train_loss'], label='Train loss')
ax1.plot(epochs, h['val_loss'],   label='Val loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Task B2 — Loss curves'); ax1.legend()

ax2.plot(epochs, h['train_acc'], label='Train acc')
ax2.plot(epochs, h['val_acc'],   label='Val acc')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Task B2 — Accuracy curves'); ax2.legend()

plt.tight_layout()
plt.savefig('b2_curves.png', dpi=120)
plt.show()
print('Saved to b2_curves.png')

In [ ]:
# Show per-layer memory audit at INT8
from task_b2_int8 import QuantizedConstrainedMGU
from utils.memory_utils import audit_model_memory
m2 = QuantizedConstrainedMGU()
audit_model_memory(m2, bytes_per_param=1)   # int8

## Section 4 — Task C: Power-of-Two Quantization-Aware Training

Weights are constrained to `{0} ∪ {±2^k | k ∈ [-8, 3]}` so that multiplications
can be replaced by bit-shifts on hardware.  Warm-starts from B2 → B1 → random.

In [ ]:
# Train Task C (60 epochs, ~15-20 min on GPU)
!python task_c_pow2.py

In [ ]:
# Plot Task C training curves from history_c.json
import json, matplotlib.pyplot as plt

with open('history_c.json') as f:
    h = json.load(f)

epochs = list(range(1, len(h['train_loss']) + 1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, h['train_loss'], label='Train loss')
ax1.plot(epochs, h['val_loss'],   label='Val loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Task C — Loss curves'); ax1.legend()

ax2.plot(epochs, h['train_acc'], label='Train acc')
ax2.plot(epochs, h['val_acc'],   label='Val acc')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Task C — Accuracy curves'); ax2.legend()

plt.tight_layout()
plt.savefig('c_curves.png', dpi=120)
plt.show()

In [ ]:
# PoT weight distribution bar chart (run after task_c_pow2.py completes)
import json, torch, collections, matplotlib.pyplot as plt
from task_c_pow2 import PoTConstrainedMGU
from utils.quant_utils import FakePoTLinear, snap_weights_to_pot
import config, copy

model_c = PoTConstrainedMGU().to('cpu')
sd = torch.load(config.CHECKPOINT_C, map_location='cpu')
model_c.load_state_dict(sd)

snapped = copy.deepcopy(model_c)
snap_weights_to_pot(snapped)

all_weights = []
for m in snapped.modules():
    if isinstance(m, FakePoTLinear):
        all_weights.append(m.linear.weight.data.flatten())

flat = torch.cat(all_weights)
counter = collections.Counter()
for v in flat.tolist():
    key = 0.0 if abs(v) < 1e-12 else round(v, 8)
    counter[key] += 1

total = flat.numel()
vals  = sorted(counter.keys())
pcts  = [100.0 * counter[v] / total for v in vals]
labels = [f"{v:.4g}" for v in vals]

plt.figure(figsize=(14, 4))
plt.bar(range(len(vals)), pcts)
plt.xticks(range(len(vals)), labels, rotation=45, ha='right', fontsize=8)
plt.ylabel('% of weights')
plt.title(f'Task C — PoT Weight Distribution ({total:,} weights)')
plt.tight_layout()
plt.savefig('c_weight_dist.png', dpi=120)
plt.show()

## Section 5 — Model Comparison

Summary table comparing all four tasks.

In [ ]:
import pandas as pd, torch, os
import config
from main import DigitGRU
from task_b1_constrained import ConstrainedMGU
from task_b2_int8 import QuantizedConstrainedMGU
from task_c_pow2 import PoTConstrainedMGU

def count_params(model):
    return sum(p.numel() for p in model.parameters())

rows = [
    {
        'Task': 'A — GRU teacher',
        'Architecture': 'GRU(hidden=128)',
        'Dtype': 'float32',
        'Params': count_params(DigitGRU(input_size=120)),
        'Per-layer limit': '36 kB',
        'Checkpoint': config.CHECKPOINT_PATH,
    },
    {
        'Task': 'B1 — MGU float32',
        'Architecture': 'MGU(proj=32, hidden=50)',
        'Dtype': 'float32',
        'Params': count_params(ConstrainedMGU()),
        'Per-layer limit': '36 kB',
        'Checkpoint': config.CHECKPOINT_B1,
    },
    {
        'Task': 'B2 — MGU INT8 QAT',
        'Architecture': 'MGU(proj=32, hidden=50)',
        'Dtype': 'int8',
        'Params': count_params(QuantizedConstrainedMGU()),
        'Per-layer limit': '36 kB',
        'Checkpoint': config.CHECKPOINT_B2,
    },
    {
        'Task': 'C — MGU PoT QAT',
        'Architecture': 'MGU(proj=32, hidden=50)',
        'Dtype': 'power-of-2',
        'Params': count_params(PoTConstrainedMGU()),
        'Per-layer limit': '36 kB',
        'Checkpoint': config.CHECKPOINT_C,
    },
]

# Add file sizes where checkpoints exist
for r in rows:
    ckpt = r['Checkpoint']
    if os.path.exists(ckpt):
        r['Checkpoint size'] = f"{os.path.getsize(ckpt) / 1024:.1f} kB"
    else:
        r['Checkpoint size'] = 'not found'

df = pd.DataFrame(rows)
df = df.set_index('Task')
print(df.to_string())
df

In [ ]:
# Download all checkpoints and training histories
from google.colab import files
import os

artifacts = [
    'best_model.pt',
    'best_model_b1_constrained.pt',
    'best_model_b2_int8.pt',
    'best_model_c_pow2.pt',
    'history_b2.json',
    'history_c.json',
    'b2_curves.png',
    'c_curves.png',
    'c_weight_dist.png',
]

for f in artifacts:
    if os.path.exists(f):
        files.download(f)
        print(f"Downloaded: {f}")
    else:
        print(f"Not found (skipped): {f}")